# Модели классификации на синтетических и реальных данных

## Требования к лабораторной работе

1. Создать «интересные» синтетические данные
2. Обучить линейную модель классификации на синтетических данных
3. Обучить полиномиальную модель классификации на синтетических данных
4. Добиться адекватных результатов модели
5. Выводы по обучению на синтетических данных
6. Выбор набора данных для обучения моделей классификации
7. Предварительная обработка датасета
8. Обучение модели логистической регрессии
9. Улучшение качества модели
10. Вывод после обучения на реальных данных
11. Выбор альтернативной модели
12. Настройка гиперпараметров
13. Сравнение с логистической регрессией
14. Общие выводы

*Использовать бустинги, ансамбли и случайный лес нельзя.*

## Библиотеки

In [1]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.datasets import make_gaussian_quantiles
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.neighbors import KNeighborsClassifier

## Константы

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

Выбор темы для графиков plotly

In [3]:
pio.templates.default = 'plotly_dark'

# Часть 1. Синтетические данные

## 1.1. Генерация данных

Выбраны набор с нелинейными границами классов:

- **make\_gaussian\_quantiles** — два класса, разделённые гауссовыми квантилами (шум 0.5).

Датасет по 1 500 объектов; разбиение 75 / 25 (обучение / валидация) со стратификацией.

In [4]:
# Гауссовы квантили
X_gaussian, y_gaussian = make_gaussian_quantiles(n_samples=1500, n_features=2, n_classes=2, random_state=RANDOM_STATE)
X_g_tr, X_g_val, y_g_tr, y_g_val = train_test_split(
    X_gaussian, y_gaussian, test_size=0.25, random_state=RANDOM_STATE, stratify=y_gaussian
)

print(f"make_gaussian_quantiles: обучение {X_g_tr.shape[0]} записей, валидация {X_g_val.shape[0]} записей")

make_gaussian_quantiles: обучение 1125 записей, валидация 375 записей


In [5]:
palette = ['#636EFA', '#EF553B']
fig = make_subplots(rows=1, cols=1,
    subplot_titles=['make_gaussian_quantiles (гауссовы квантили)'])

shown = set()
for col, (X_tr, y_tr, X_v, y_v) in enumerate([
    (X_g_tr, y_g_tr, X_g_val, y_g_val)
], 1):
    for cls in [0, 1]:
        for name, X_s, y_s, sym in [('обучение', X_tr, y_tr, 'circle'),
                                     ('валидация', X_v,  y_v,  'diamond')]:
            label = f'Класс {cls} — {name}'
            mask  = y_s == cls
            fig.add_trace(go.Scatter(
                x=X_s[mask, 0], y=X_s[mask, 1], mode='markers',
                name=label, legendgroup=label, showlegend=(label not in shown),
                marker=dict(color=palette[cls], symbol=sym, size=6, opacity=0.7,
                            line=dict(width=0.5, color='white'))
            ), row=1, col=col)
            shown.add(label)

fig.update_layout(title='Рисунок 1. Синтетический набор данных', height=460, width=1000,
                  xaxis_title='x₁', yaxis_title='x₂')
fig.show()

## 1.2. Логистическая регрессия и полиномы

`make_gaussian_quantiles` формирует **радиальную (эллиптическую) границу** классов:
класс 0 — внутренняя область, класс 1 — внешняя. Линейная граница заведомо не справится,
поскольку разделяющая поверхность — изолиния $x_1^2 + x_2^2 = \text{const}$.

Сравниваются степени 1, 2, 3, 5, 8, 10, 12, 15 с масштабированием и L2-регуляризацией.

In [6]:
def make_pipe(degree, C=1.0):
    steps = []
    if degree > 1:
        steps.append(('poly', PolynomialFeatures(degree=degree, include_bias=False)))
    steps += [('scaler', StandardScaler()),
              ('clf', LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE))]
    return Pipeline(steps)

def plot_boundary(model, X_tr, y_tr, X_v, y_v, title, n=220):
    x0 = np.linspace(X_tr[:,0].min()-0.5, X_tr[:,0].max()+0.5, n)
    x1 = np.linspace(X_tr[:,1].min()-0.5, X_tr[:,1].max()+0.5, n)
    xx0, xx1 = np.meshgrid(x0, x1)
    Z = model.predict_proba(np.c_[xx0.ravel(), xx1.ravel()])[:,1].reshape(xx0.shape)
    fig = go.Figure()
    fig.add_trace(go.Contour(x=x0, y=x1, z=Z, colorscale='RdBu_r', opacity=0.45,
                             contours=dict(showlines=False), showscale=True,
                             colorbar=dict(title='Контур вероятности для кл.1', len=0.7, orientation='v', yanchor='top', y=0.7)))
    for cls in [0, 1]:
        for nm, X_s, y_s, sym in [('обуч.', X_tr, y_tr, 'circle'),
                                   ('вал.', X_v, y_v, 'diamond-open')]:
            mask = y_s == cls
            fig.add_trace(go.Scatter(x=X_s[mask,0], y=X_s[mask,1], mode='markers',
                name=f'кл.{cls} {nm}', marker=dict(color=palette[cls], symbol=sym, size=6)))
    fig.update_layout(title=title, width=1000, height=460, xaxis_title='x₁', yaxis_title='x₂',
                      legend=dict(font=dict(size=10)))
    return fig

In [7]:
# Линейная граница (degree=1)
pipe_lin = make_pipe(1)
pipe_lin.fit(X_g_tr, y_g_tr)
acc_tr = pipe_lin.score(X_g_tr, y_g_tr)
acc_v  = pipe_lin.score(X_g_val, y_g_val)
print(f'Линейная модель: обучение={acc_tr:.3f}, валидация={acc_v:.3f}')
plot_boundary(pipe_lin, X_g_tr, y_g_tr, X_g_val, y_g_val,
              f'Рисунок 2. Линейная граница (make_gaussian_quantiles) | val acc={acc_v:.3f}').show()

Линейная модель: обучение=0.534, валидация=0.512


In [8]:
# Полиномы разных степеней — сбор метрик
degrees = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
rows = []
pipes = {}

for d in degrees:
    p = make_pipe(d)
    p.fit(X_g_tr, y_g_tr)
    pipes[d] = p
    n_feat = p.named_steps['clf'].coef_.shape[1]
    rows.append(dict(degree=d,
                     train=p.score(X_g_tr, y_g_tr),
                     val=p.score(X_g_val, y_g_val),
                     n_feat=n_feat,
                     coef_norm=float(np.linalg.norm(p.named_steps['clf'].coef_))))

res = pd.DataFrame(rows)
print(res.to_string(index=False))

 degree    train      val  n_feat  coef_norm
      1 0.534222 0.512000       2   0.112311
      2 0.981333 0.984000       5   9.712620
      3 0.982222 0.986667       9   9.721287
      4 0.981333 0.978667      14   9.520251
      5 0.981333 0.978667      20   9.517987
      6 0.981333 0.978667      27   9.512528
      7 0.981333 0.978667      35   9.510788
      8 0.981333 0.978667      44   9.503760
      9 0.981333 0.978667      54   9.508263
     10 0.981333 0.978667      65   9.507251
     11 0.981333 0.978667      77   9.499291
     12 0.981333 0.978667      90   9.505589
     13 0.981333 0.978667     104   9.504614
     14 0.981333 0.978667     119   9.518716
     15 0.981333 0.978667     135   9.527632


In [9]:
# Кривые точности по степени
fig = go.Figure()
fig.add_trace(go.Scatter(x=res['degree'], y=res['train'],
    mode='lines+markers', name='Обучение', line=dict(color='#636EFA')))
fig.add_trace(go.Scatter(x=res['degree'], y=res['val'],
    mode='lines+markers', name='Валидация', line=dict(color='#EF553B')))
fig.update_layout(title='Рисунок 3. Точность по степени полинома (make_gaussian_quantiles)',
    xaxis_title='Степень', yaxis_title='Accuracy', height=400, width=800)
fig.show()

In [10]:
# Границы решений для степеней 2, 3, 10
for d in [2, 3, 10]:
    iteration = [2, 3, 10].index(d)
    v = pipes[d].score(X_g_val, y_g_val)
    plot_boundary(pipes[d], X_g_tr, y_g_tr, X_g_val, y_g_val,
                  f'Рисунок {4 + iteration}. Полином степени {d} (make_gaussian_quantiles) | val acc={v:.3f}').show()

## 1.3. Артефакты поведения модели

Анализ проводится на **малой выборке** (200 точек из `make_gaussian_quantiles`)
с ослабленной регуляризацией (C=1000), чтобы усилить эффекты.

**Переобучение**: при degree=1 модель не способна выделить круговую границу;
при degree=2 качество резко возрастает; при степенях выше (≥ 4) и малом n
обучающая точность стремится к 1, а максимум валидационной точности не растёт.

**Резкий рост нормы коэффициентов** ||w||₂ (евклидова длина) — при переходе от degree=1 к degree=2
норма вырастает на порядок, коэффициенты соответствуют нечитаемым
кросс-произведениям вроде $x_1^3 x_2^2$.

In [11]:
# Артефакт 1: переобучение — малая выборка (200 точек) + слабая регуляризация (C=1000)
X_small, y_small = X_gaussian[:200], y_gaussian[:200]
X_sv,    y_sv    = X_gaussian[200:400], y_gaussian[200:400]

degrees_art = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
rows_art = []
for d in degrees_art:
    p = make_pipe(d, C=1000)
    p.fit(X_small, y_small)
    rows_art.append(dict(
        degree=d,
        train=p.score(X_small, y_small),
        val=p.score(X_sv, y_sv),
        coef_norm=float(np.linalg.norm(p.named_steps['clf'].coef_))
    ))

art_df = pd.DataFrame(rows_art)
print(art_df.to_string(index=False))

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Переобучение: train vs val (n=200, C=1000)',
                    'Норма коэффициентов ||w||₂ (n=200, C=1000)'])

fig.add_trace(go.Scatter(x=art_df['degree'], y=art_df['train'],
    mode='lines+markers', name='Обучение', line=dict(color='#636EFA')), row=1, col=1)
fig.add_trace(go.Scatter(x=art_df['degree'], y=art_df['val'],
    mode='lines+markers', name='Валидация', line=dict(color='#EF553B')), row=1, col=1)
fig.add_trace(go.Scatter(x=art_df['degree'], y=art_df['coef_norm'],
    mode='lines+markers', name='||w||₂', line=dict(color='#00CC96')), row=1, col=2)

for col in [1, 2]:
    fig.update_xaxes(title_text='Степень', row=1, col=col)
fig.update_yaxes(title_text='Accuracy', row=1, col=1)
fig.update_yaxes(title_text='||w||₂', row=1, col=2)
fig.update_layout(title='Рисунок 7. Артефакты полиномиальной логистической регрессии', height=420)
fig.show()

 degree  train   val  coef_norm
      1   0.66 0.620   0.235642
      2   1.00 0.985  53.246081
      3   1.00 0.975  52.416100
      4   1.00 0.980  50.767893
      5   1.00 0.975  51.345343
      6   1.00 0.980  51.316274
      7   1.00 0.980  52.499471
      8   1.00 0.980  51.776232
      9   1.00 0.975  52.605028
     10   1.00 0.980  52.302911
     11   1.00 0.980  52.138849
     12   1.00 0.975  52.105549
     13   1.00 0.975  51.770165
     14   1.00 0.975  50.709164
     15   1.00 0.975  51.810558


In [12]:
# Влияние регуляризации при degree=10
C_vals = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 20.0]
reg_rows = []
for C in C_vals:
    p = make_pipe(10, C=C)
    p.fit(X_g_tr, y_g_tr)
    reg_rows.append(dict(C=C, log_C=float(np.log10(C)),
                         train=p.score(X_g_tr, y_g_tr),
                         val=p.score(X_g_val, y_g_val),
                         coef_max=float(np.abs(p.named_steps['clf'].coef_).max())))
reg_df = pd.DataFrame(reg_rows)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Точность vs log₁₀(C)', 'Макс. |коэффициент| vs log₁₀(C)'])
fig.add_trace(go.Scatter(x=reg_df['log_C'], y=reg_df['train'], mode='lines+markers',
    name='Обучение', line=dict(color='#636EFA')), row=1, col=1)
fig.add_trace(go.Scatter(x=reg_df['log_C'], y=reg_df['val'], mode='lines+markers',
    name='Валидация', line=dict(color='#EF553B')), row=1, col=1)
fig.add_trace(go.Scatter(x=reg_df['log_C'], y=reg_df['coef_max'], mode='lines+markers',
    name='|coef|_max', line=dict(color='#AB63FA')), row=1, col=2)
fig.update_xaxes(title_text='log₁₀(C)', row=1, col=1)
fig.update_xaxes(title_text='log₁₀(C)', row=1, col=2)
fig.update_yaxes(title_text='Accuracy', row=1, col=1)
fig.update_yaxes(title_text='|coef|_max', row=1, col=2)
fig.update_layout(title='Рисунок 8. Влияние регуляризации при степени 10 (make_gaussian_quantiles)',
                  height=420, width=1000)
fig.show()

## 1.4. Выводы по синтетическому эксперименту

**Соответствие требованиям**: `make_gaussian_quantiles` формирует классы по квантилям
многомерного нормального распределения. Данный метод отличен от демострационных примеров с полумесяцами и кругами (вне этого блокнота).

- **Линейная граница полностью бесполезна** (`accuracy ≈ 50 %` — уровень случайного угадывания):
  прямая не может выделить внутреннюю область Гауссова распределения.
- **Степень 2** даёт резкий скачок качества: одночлены $x_1^2$, $x_2^2$ и $x_1 x_2$
  достаточны для аппроксимации круговой границы. Это наглядно показывает, зачем нужны
  полиномиальные признаки.
- Степени 3 добавляtт несколько **десятых** процентов точности — граница уже хорошо описана
  квадратичными членами, остальные улавливают лишь нелинейности в шуме.
- При степени ≥ 4 и слабой регуляризации (C=1000, n=200) — **это переобучение**:
  train → 1.0, val падает; норма коэффициентов ||w||₂ вырастает на порядок.
- **Регуляризация** при degree=10 удерживает val-качество: оптимальная зона
  $C \in [0.1, 1.0]$. При $C < 0.01$ (log₁₀(C) < -2) модель недообучается, при $C > 5$ (log₁₀(C) > 0.7) — переобучается.
- Коэффициенты при степенях ≥ 3 соответствуют кросс-произведениям типа $x_1^2 x_2$
  — их физическая интерпретация невозможна.

# Часть 2. Реальные данные. Линейная модель

## 2.1. Выбор набора данных

**Diabetes Health Indicators Dataset** (Kaggle, alexteboul) —
253 680 ответов опроса BRFSS 2015, очищенных и бинаризованных.
Используется **сбалансированная версия** (50/50):
`diabetes_binary_5050split_health_indicators_BRFSS2015.csv`.

- 21 признак: артериальное давление, холестерин, ИМТ, курение, физическая активность,
  общее состояние здоровья, возраст, образование, доход и др.
- Целевая переменная `Diabetes_binary`: 0 — нет диабета, 1 — есть диабет (оба типа).
- **Задача**: бинарная классификация.

##  2.2. Загрузка данных
Загрузка датасета по такому же пути как в первой лабораторной работе, через kaggle API.

In [13]:
# Перед запуском убедитесь в установке python на компьютере
%pip install kaggle nbformat # Установка зависимостей для работы с Kaggle API и Jupyter Notebook

Note: you may need to restart the kernel to use updated packages.


In [14]:
DATA_FILE = 'diabetes_binary_5050split_health_indicators_BRFSS2015.csv'

if not os.path.exists(DATA_FILE):
    # Переменная KAGGLE_API_TOKEN должна быть установлена в переменных окружения для доступа к Kaggle API
    # Инструкция по созданию токена в README.md Пункт 5
    # Скачиваем датасет через Kaggle CLI shell-командой
    !kaggle datasets download --unzip --force alexteboul/diabetes-health-indicators-dataset # скачиваем датасет через kaggle CLI
    
    # В репозитории 3 файла, но потребуется только diabetes_binary_5050split_health_indicators_BRFSS2015.csv

df = pd.read_csv(DATA_FILE)
print(f"Размер: {df.shape[0]} строк × {df.shape[1]} столбцов")

number_of_columns_to_show = 5
print(f"Первые {number_of_columns_to_show} строк датасета:")
df.head(number_of_columns_to_show)

Размер: 70692 строк × 22 столбцов
Первые 5 строк датасета:


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,8.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,8.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,8.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,8.0


## 4.2. Предварительная обработка

Все признаки в датасете числовые (бинарные или порядковые) — специального кодирования
не требуется. Проверяются пропуски и дубликаты, строится базовая статистика.

In [15]:
print('=== Пропуски ===')
print(df.isnull().sum().to_string())
print(f'\n=== Дубликаты: {df.duplicated().sum()} строк ===')
print(f'\n=== Целевая переменная ===')
print(df['Diabetes_binary'].value_counts())

=== Пропуски ===
Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0

=== Дубликаты: 1635 строк ===

=== Целевая переменная ===
Diabetes_binary
0.0    35346
1.0    35346
Name: count, dtype: int64


In [16]:
df.describe()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,...,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000
mean,0.500000,0.563458,0.525703,0.975259,29.856985,0.475273,0.062171,0.147810,0.703036,0.611795,...,0.954960,0.093914,2.837082,3.752037,5.810417,0.252730,0.456997,8.584055,4.920953,5.698311
std,0.500004,0.495960,0.499342,0.155336,7.113954,0.499392,0.241468,0.354914,0.456924,0.487345,...,0.207394,0.291712,1.113565,8.155627,10.062261,0.434581,0.498151,2.852153,1.029081,2.175196
min,0.000000,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,7.000000,4.000000,4.000000
50%,0.500000,1.000000,1.000000,1.000000,29.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,9.000000,5.000000,6.000000
75%,1.000000,1.000000,1.000000,1.000000,33.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,4.000000,2.000000,6.000000,1.000000,1.000000,11.000000,6.000000,8.000000
max,1.000000,1.000000,1.000000,1.000000,98.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000


In [17]:
# Группировка по классу диабета и средние значения признаков
df.groupby('Diabetes_binary').mean().T

Diabetes_binary,0.0,1.0
HighBP,0.374243,0.752674
HighChol,0.381288,0.670118
CholCheck,0.957336,0.993182
BMI,27.769960,31.944011
Smoker,0.432326,0.518220
Stroke,0.031885,0.092457
HeartDiseaseorAttack,0.072738,0.222882
PhysActivity,0.775533,0.630538
Fruits,0.638149,0.585441
Veggies,0.821140,0.756408


In [18]:
# Распределение целевой переменной
cnt = df['Diabetes_binary'].value_counts().reset_index()
cnt.columns = ['Класс', 'Количество']
cnt['Класс'] = cnt['Класс'].map({0: 'Нет диабета (0)', 1: 'Есть диабет (1)'})

fig = px.bar(cnt, x='Класс', y='Количество', color='Класс',
             color_discrete_map={'Нет диабета (0)': '#636EFA', 'Есть диабет (1)': '#EF553B'},
             title='Рисунок 9. Распределение целевой переменной',
             text='Количество')
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=400, width=600)
fig.show()

In [19]:
# Распределение числовых признаков по классам
cont_cols = ['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']
fig = make_subplots(rows=2, cols=4, subplot_titles=cont_cols)

for i, col in enumerate(cont_cols):
    r, c = divmod(i, 4)
    for cls, color, nm in [(0, '#636EFA', 'Нет диабета'), (1, '#EF553B', 'Есть диабет')]:
        fig.add_trace(go.Histogram(
            x=df[df['Diabetes_binary']==cls][col], name=nm,
            legendgroup=nm, showlegend=(i==0),
            marker_color=color, opacity=0.6, nbinsx=20
        ), row=r+1, col=c+1)

fig.update_layout(barmode='overlay', height=500, width=1200,
    title='Рисунок 10. Распределение числовых признаков по наличию диабета')
fig.show()

In [20]:
# Корреляция признаков с целевой переменной
target_corr = df.corr(numeric_only=True)['Diabetes_binary'].drop('Diabetes_binary')
target_corr = target_corr.sort_values()

fig = px.bar(x=target_corr.values, y=target_corr.index, orientation='h',
             color=target_corr.values, color_continuous_scale='RdBu_r',
             title='Рисунок 11. Корреляция Пирсона признаков с Diabetes_binary',
             labels={'x': 'Корреляция', 'y': 'Признак'})
fig.update_layout(height=560, width=800, coloraxis_showscale=False)
fig.show()

In [21]:
TARGET   = 'Diabetes_binary'
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Обучение: {X_train.shape[0]:,}  |  Тест: {X_test.shape[0]:,}')

Обучение: 56,553  |  Тест: 14,139


## 4.3. Базовая логистическая регрессия

Базовая (baseline) модель: только масштабирование и логистическая регрессия
с параметрами по умолчанию. Позволяет понять нижнюю границу качества и
увидеть характерные ошибки.

In [22]:
pipe_base = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
pipe_base.fit(X_train, y_train)

y_pred_base = pipe_base.predict(X_test)
y_prob_base = pipe_base.predict_proba(X_test)[:, 1]

def print_metrics(name, y_true, y_pred, y_prob):
    print(f'── {name} ──')
    print(f'  Accuracy : {accuracy_score(y_true, y_pred):.4f}')
    print(f'  Precision: {precision_score(y_true, y_pred):.4f}')
    print(f'  Recall   : {recall_score(y_true, y_pred):.4f}')
    print(f'  F1       : {f1_score(y_true, y_pred):.4f}')
    print(f'  ROC-AUC  : {roc_auc_score(y_true, y_prob):.4f}')

print_metrics('Базовая логистическая регрессия', y_test, y_pred_base, y_prob_base)

── Базовая логистическая регрессия ──
  Accuracy : 0.7458
  Precision: 0.7372
  Recall   : 0.7639
  F1       : 0.7503
  ROC-AUC  : 0.8232


In [23]:
def plot_cm(y_true, y_pred, title):
    cm     = confusion_matrix(y_true, y_pred)
    labels = ['Нет диабета', 'Есть диабет']
    fig = px.imshow(cm, x=labels, y=labels, text_auto=True,
                    color_continuous_scale='Blues', aspect='auto',
                    labels=dict(x='Предсказание', y='Истина', color='Количество'),
                    title=title)
    fig.update_layout(height=380, width=650)
    return fig

plot_cm(y_test, y_pred_base, 'Рисунок 12. Матрица ошибок — базовая логистическая регрессия').show()

Матрица ошибок показывает, что модель примерно поровну ошибается в обоих направлениях
(FP и FN сопоставимы), что ожидаемо на сбалансированных данных.
Заметно, что **Recall** чуть выше **Precision** — модель склонна слегка
недооценивать число здоровых.

In [24]:
# Важность признаков по модулю коэффициента
coef = pipe_base.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'Признак': FEATURES, 'Коэффициент': coef})
coef_df = coef_df.reindex(coef_df['Коэффициент'].abs().sort_values(ascending=False).index)

fig = px.bar(coef_df, x='Коэффициент', y='Признак', orientation='h',
             color='Коэффициент', color_continuous_scale='RdBu_r',
             title='Рисунок 13. Важность признаков по |коэффициенту| (базовая логистическая регрессия)')
fig.update_layout(height=480, width=800,
                  yaxis=dict(autorange='reversed'), coloraxis_showscale=False)
fig.show()

## 4.4. Улучшение качества модели

Проверяется влияние силы регуляризации (параметр C) через 5-кратную
кросс-валидацию по ROC-AUC. Дополнительно тестируется SGDClassifier с функцией потерь
`log_loss` — стохастический аналог логистической регрессии, устойчивый к большим данным.

In [25]:
# Поиск лучшего C
C_grid = [0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
cv_rows = []

for C in C_grid:
    p = Pipeline([('scaler', StandardScaler()),
                  ('clf', LogisticRegression(C=C, max_iter=1000, random_state=RANDOM_STATE))])
    scores = cross_val_score(p, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    cv_rows.append(dict(C=C, log_C=float(np.log10(C)),
                        mean=scores.mean(), std=scores.std()))

cv_df = pd.DataFrame(cv_rows)
best_C = cv_df.loc[cv_df['mean'].idxmax(), 'C']
print(f"Лучшее C = {best_C}  (ROC-AUC CV = {cv_df['mean'].max():.4f})")

fig = go.Figure()
fig.add_trace(go.Scatter(x=cv_df['log_C'], y=cv_df['mean'], mode='lines+markers',
    name='ROC-AUC (CV=5)',
    error_y=dict(type='data', array=cv_df['std'].tolist(), visible=True)))
fig.update_layout(title='Рисунок 14. Кросс-валидационный ROC-AUC vs C (логистическая регрессия)',
    xaxis_title='log₁₀(C)', yaxis_title='ROC-AUC', height=400, width=700)
fig.show()

Лучшее C = 0.01  (ROC-AUC CV = 0.8251)


In [26]:
# Лучшая логистическая регрессия
print(type(best_C))
pipe_lr_best = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=best_C, max_iter=1000, random_state=RANDOM_STATE))
])
pipe_lr_best.fit(X_train, y_train)

y_pred_lr = pipe_lr_best.predict(X_test)
y_prob_lr = pipe_lr_best.predict_proba(X_test)[:, 1]
print_metrics('Логистическая регрессия (лучшая)', y_test, y_pred_lr, y_prob_lr)

<class 'numpy.float64'>
── Логистическая регрессия (лучшая) ──
  Accuracy : 0.7457
  Precision: 0.7370
  Recall   : 0.7640
  F1       : 0.7503
  ROC-AUC  : 0.8232


In [27]:
# SGDClassifier (аналог логистической регрессии — стохастический градиент)
pipe_sgd = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    SGDClassifier(loss='log_loss', alpha=1e-3, max_iter=500,
                             random_state=RANDOM_STATE, n_jobs=-1))
])
pipe_sgd.fit(X_train, y_train)

y_pred_sgd = pipe_sgd.predict(X_test)
y_prob_sgd = pipe_sgd.predict_proba(X_test)[:, 1]
print_metrics('SGDClassifier (log_loss)', y_test, y_pred_sgd, y_prob_sgd)

── SGDClassifier (log_loss) ──
  Accuracy : 0.7428
  Precision: 0.7372
  Recall   : 0.7547
  F1       : 0.7458
  ROC-AUC  : 0.8221


## 4.5. Выводы по реальным данным и предсказания на новых примерах

- Обе версии логистической регрессии (базовая и с подобранным C) дают практически
  одинаковые результаты: ROC-AUC ≈ **0,823**, Accuracy ≈ **0,746** — подбор C
  не даёт заметного прироста на сбалансированных данных.
- Ключевые предикторы: **GenHlth** (общее самочувствие), **BMI**, **Age**, **HighBP**.
- Матрица ошибок симметрична: FP и FN сопоставимы, что ожидаемо на сбалансированной выборке.
- SGDClassifier показывает сопоставимое качество (ROC-AUC ≈ 0,822) и удобен для потоковой обработки.

Ниже — предсказания на 10 составленных примерах с разным профилем риска.

In [28]:
with pd.option_context('display.max_columns', None, 'display.width', None):
    display(df.head(1))

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0


In [29]:
# 10 составленных примеров
# Порядок: HighBP HighChol CholCheck BMI Smoker Stroke HeartDiseaseorAttack
#           PhysActivity Fruits Veggies HvyAlcoholConsump AnyHealthcare
#           NoDocbcCost GenHlth MentHlth PhysHlth DiffWalk Sex Age Education Income

FEATURES_WITH_DESCRIPTION = ['Описание', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke',
                             'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
                             'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
                             'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']

tricky_examples = pd.DataFrame([
    # Описание                                          BP HC Ch  BMI  Sm St HD  PA Fr Ve Al HC ND GH  Me  Ph DW Sx  Ag Ed In
    ["молодой и здоровый",                              0, 0, 1,  22,  0, 0, 0,  1, 1, 1, 0, 1, 0, 1,  0,  0, 0, 0,  2, 6, 8],  # молодой, здоровый
    ["пожилой, много рисков",                           1, 1, 1,  38,  1, 1, 1,  0, 0, 0, 0, 1, 0, 5, 20, 25, 1, 1, 12, 3, 2],  # пожилой, много рисков
    ["очень молодой, не обследовался",                  0, 0, 0,  19,  0, 0, 0,  1, 1, 1, 0, 0, 1, 2,  5,  3, 0, 0,  1, 5, 7],  # очень молодой, не обследовался
    ["средний возраст, лишний вес",                     1, 1, 1,  32,  0, 0, 0,  0, 1, 1, 0, 1, 0, 4, 10, 15, 0, 1,  9, 4, 4],  # ср. возраст, лишний вес
    ["алкоголик, активный",                             0, 0, 1,  25,  0, 0, 0,  1, 1, 1, 1, 1, 0, 2,  0,  0, 0, 1,  4, 6, 7],  # умер. алкоголь, активный
    ["пожилая, инсульт, ожирение",                      1, 1, 1,  40,  1, 1, 1,  0, 0, 0, 0, 1, 1, 5, 15, 28, 1, 0, 11, 2, 1],  # пожилая, инсульт, ожирение
    ["средний возраст, курит, активен",                 1, 0, 1,  28,  1, 0, 0,  1, 1, 0, 0, 1, 0, 3,  5, 10, 0, 0,  7, 5, 5],  # ср. возраст, курит, активен
    ["молодой, очень здоровый",                         0, 0, 1,  21,  0, 0, 0,  1, 1, 1, 0, 1, 0, 1,  0,  0, 0, 1,  3, 6, 8],  # молодой, очень здоровый
    ["пожилой, сердечно-сосудистые заболевания (ССЗ)",  1, 1, 1,  35,  0, 1, 1,  0, 0, 0, 0, 1, 0, 5, 20, 20, 1, 1, 10, 3, 2],  # пожилой, сердечно-сосудистые заболевания (ССЗ)
    ["средний возраст, курит, не следит за здоровьем",  0, 1, 1,  24,  1, 0, 0,  0, 0, 0, 0, 1, 1, 3, 10,  5, 0, 0,  5, 4, 5],  # ср. возраст, курит, не следит за здоровьем
], columns=FEATURES_WITH_DESCRIPTION)

with pd.option_context('display.max_columns', None, 'display.width', None):
    display(tricky_examples)

,Описание,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,молодой и здоровый,0,0,1,22,0,0,0,1,1,1,0,1,0,1,0,0,0,0,2,6,8
1,"пожилой, много рисков",1,1,1,38,1,1,1,0,0,0,0,1,0,5,20,25,1,1,12,3,2
2,"очень молодой, не обследовался",0,0,0,19,0,0,0,1,1,1,0,0,1,2,5,3,0,0,1,5,7
3,"средний возраст, лишний вес",1,1,1,32,0,0,0,0,1,1,0,1,0,4,10,15,0,1,9,4,4
4,"алкоголик, активный",0,0,1,25,0,0,0,1,1,1,1,1,0,2,0,0,0,1,4,6,7
5,"пожилая, инсульт, ожирение",1,1,1,40,1,1,1,0,0,0,0,1,1,5,15,28,1,0,11,2,1
6,"средний возраст, курит, активен",1,0,1,28,1,0,0,1,1,0,0,1,0,3,5,10,0,0,7,5,5
7,"молодой, очень здоровый",0,0,1,21,0,0,0,1,1,1,0,1,0,1,0,0,0,1,3,6,8
8,"пожилой, сердечно-сосудистые заболевания (ССЗ)",1,1,1,35,0,1,1,0,0,0,0,1,0,5,20,20,1,1,10,3,2
9,"средний возраст, курит, не следит за здоровьем",0,1,1,24,1,0,0,0,0,0,0,1,1,3,10,5,0,0,5,4,5


In [30]:
tricky_examples = tricky_examples.drop(['Описание'], axis=1)

probs_lr = pipe_lr_best.predict_proba(tricky_examples[FEATURES].values)[:, 1]
preds_lr = pipe_lr_best.predict(tricky_examples[FEATURES].values)

descriptions = [
    'Молодой, здоровый', 'Пожилой, много рисков', 'Очень молодой, не обследовался',
    'Ср. возраст, лишний вес', 'пьет алкоголь, активный', 'Пожилая, инсульт, ожирение',
    'Ср. возраст, курит, активен', 'Молодой, очень здоровый',
    'Пожилой, ССЗ', 'Ср. возраст, курит'
]

res_lr = pd.DataFrame({
    'Описание':    descriptions,
    'P(диабет)':   probs_lr.round(3),
    'Прогноз ЛР':  ['Диабет' if p == 1 else 'Норма' for p in preds_lr]
})
print(res_lr.to_string(index=False))

                      Описание  P(диабет) Прогноз ЛР
             Молодой, здоровый      0.025      Норма
         Пожилой, много рисков      0.977     Диабет
Очень молодой, не обследовался      0.009      Норма
       Ср. возраст, лишний вес      0.837     Диабет
       пьет алкоголь, активный      0.048      Норма
    Пожилая, инсульт, ожирение      0.974     Диабет
   Ср. возраст, курит, активен      0.394      Норма
       Молодой, очень здоровый      0.035      Норма
                  Пожилой, ССЗ      0.964     Диабет
            Ср. возраст, курит      0.263      Норма


In [31]:
fig = px.bar(res_lr, x='Описание', y='P(диабет)',
             color='P(диабет)', color_continuous_scale='RdYlGn_r',
             title='Рисунок 15. Вероятность диабета — логистическая регрессия (новые примеры)',
             text=res_lr['P(диабет)'].round(3))
fig.add_hline(y=0.5, line_dash='dash', line_color='magenta',
              annotation_text='Порог 0.5', annotation_position='top right')
fig.update_traces(textposition='outside')
fig.update_layout(height=450, width=1200, xaxis_tickangle=-25, coloraxis_showscale=False,
                  xaxis_title='', yaxis_title='P(Diabetes_binary = 1)')
fig.show()

# Часть 3. Альтернативная модель — метод k ближайших соседей

## 5.1. Выбор модели

Выбран **k-NN** (KNeighborsClassifier). Метод непараметрический: решение принимается
на основе голосования среди $k$ ближайших обучающих примеров.

Преимущества: нет предположений о форме границы, просто интерпретируется.
Недостаток: медленно на больших выборках, чувствителен к масштабу признаков
(поэтому перед ним ставится StandardScaler).

In [32]:
# Базовый k-NN (k=5)
pipe_knn5 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    KNeighborsClassifier(n_neighbors=5))
])
pipe_knn5.fit(X_train, y_train)

y_pred_knn5 = pipe_knn5.predict(X_test)
y_prob_knn5 = pipe_knn5.predict_proba(X_test)[:, 1]
print_metrics('k-NN (k=5, базовый)', y_test, y_pred_knn5, y_prob_knn5)

── k-NN (k=5, базовый) ──
  Accuracy : 0.7126
  Precision: 0.7020
  Recall   : 0.7387
  F1       : 0.7199
  ROC-AUC  : 0.7714


## 5.2. Настройка гиперпараметров

GridSearchCV по трём параметрам: числу соседей, метрике расстояния и весовой схеме.
Для ускорения используется 3-кратная CV и параллельное вычисление (`n_jobs=-1`).

$$\text{KNN: } \hat{y} = \underset{c}{\arg\max} \sum_{i \in N_k(x)} w_i \cdot \mathbf{1}[y_i = c] \tag{1}$$

где $w_i = 1$ (uniform) или $w_i = 1/d_i$ (distance).

In [33]:
param_grid = {
    'clf__n_neighbors': [5, 10, 15, 20, 30, 50],
    'clf__metric':      ['euclidean', 'manhattan'],
    'clf__weights':     ['uniform', 'distance']
}

pipe_knn_gs = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    KNeighborsClassifier())
])

gs = GridSearchCV(pipe_knn_gs, param_grid, cv=3, scoring='roc_auc',
                  n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print(f'Лучшие параметры: {gs.best_params_}')
print(f'ROC-AUC на CV:    {gs.best_score_:.4f}')

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Лучшие параметры: {'clf__metric': 'manhattan', 'clf__n_neighbors': 50, 'clf__weights': 'uniform'}
ROC-AUC на CV:    0.8166


In [34]:
knn_best = gs.best_estimator_
y_pred_knn = knn_best.predict(X_test)
y_prob_knn = knn_best.predict_proba(X_test)[:, 1]
print_metrics(f'k-NN (лучший)', y_test, y_pred_knn, y_prob_knn)

── k-NN (лучший) ──
  Accuracy : 0.7414
  Precision: 0.7270
  Recall   : 0.7732
  F1       : 0.7494
  ROC-AUC  : 0.8184


In [35]:
# Как меняется ROC-AUC с k (euclidean + uniform и manhattan + distance)
cv_res = pd.DataFrame(gs.cv_results_)

fig = go.Figure()
for metric, weights, color, nm in [
    ('euclidean', 'uniform',  '#636EFA', 'euclidean + uniform'),
    ('manhattan', 'distance', '#EF553B', 'manhattan + distance'),
]:
    sub = cv_res[
        (cv_res['param_clf__metric']  == metric) &
        (cv_res['param_clf__weights'] == weights)
    ].sort_values('param_clf__n_neighbors')
    fig.add_trace(go.Scatter(
        x=sub['param_clf__n_neighbors'].tolist(),
        y=sub['mean_test_score'].tolist(),
        mode='lines+markers', name=nm, line=dict(color=color),
        error_y=dict(type='data', array=sub['std_test_score'].tolist(), visible=True)
    ))

fig.update_layout(title='Рисунок 16. ROC-AUC k-NN в зависимости от числа соседей',
    xaxis_title='Число соседей k', yaxis_title='ROC-AUC (3-fold CV)', height=700, width=800)
fig.show()

## 5.3. Сравнение с логистической регрессией

In [36]:
# Итоговая таблица метрик
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']

def get_metrics(y_true, y_pred, y_prob):
    return [accuracy_score(y_true, y_pred),
            precision_score(y_true, y_pred),
            recall_score(y_true, y_pred),
            f1_score(y_true, y_pred),
            roc_auc_score(y_true, y_prob)]

compare = pd.DataFrame({
    'Метрика':                   metric_names,
    'Лог. регрессия (базовая)':  get_metrics(y_test, y_pred_base, y_prob_base),
    'Лог. регрессия (лучшая)':   get_metrics(y_test, y_pred_lr,   y_prob_lr),
    'k-NN (лучший)':             get_metrics(y_test, y_pred_knn,  y_prob_knn),
}).round(4)

print(compare.to_string(index=False))

  Метрика  Лог. регрессия (базовая)  Лог. регрессия (лучшая)  k-NN (лучший)
 Accuracy                    0.7458                   0.7457         0.7414
Precision                    0.7372                   0.7370         0.7270
   Recall                    0.7639                   0.7640         0.7732
       F1                    0.7503                   0.7503         0.7494
  ROC-AUC                    0.8232                   0.8232         0.8184


In [37]:
# Групповая гистограмма метрик
models = ['Лог. регрессия (базовая)', 'Лог. регрессия (лучшая)', 'k-NN (лучший)']
fig = go.Figure()
for model in models:
    fig.add_trace(go.Bar(name=model, x=metric_names, y=compare[model].tolist()))

fig.update_layout(barmode='group', title='Рисунок 17. Сравнение моделей по метрикам',
    yaxis_title='Значение', yaxis_range=[0.6, 0.82], height=440, width=800)
fig.show()

In [38]:
# Матрицы ошибок рядом
labels = ['Нет диабета', 'Есть диабет']
cm_lr  = confusion_matrix(y_test, y_pred_lr)
cm_knn = confusion_matrix(y_test, y_pred_knn)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Логистическая регрессия (лучшая)', 'k-NN (лучший)'])
for col, cm in enumerate([cm_lr, cm_knn], 1):
    fig.add_trace(go.Heatmap(z=cm, x=labels, y=labels, colorscale='Blues',
        text=cm, texttemplate='%{text}', showscale=False,
        hovertemplate='Истина: %{y}<br>Предсказание: %{x}<br>Кол-во: %{z}<extra></extra>'
    ), row=1, col=col)

fig.update_yaxes(autorange='reversed')
fig.update_layout(height=420, width=1200, title='Рисунок 18. Матрицы ошибок на тестовой выборке')
fig.show()

In [39]:
# Предсказания k-NN на тех же 10 примерах
probs_knn_new = knn_best.predict_proba(tricky_examples[FEATURES].values)[:, 1]
preds_knn_new = knn_best.predict(tricky_examples[FEATURES].values)

res_compare = pd.DataFrame({
    'Описание':       descriptions,
    'P LogisticRegression':           probs_lr.round(3),
    'Прогноз LogisticRegression':     ['Диабет' if p == 1 else 'Норма' for p in preds_lr],
    'P k-NN':         probs_knn_new.round(3),
    'Прогноз k-NN':   ['Диабет' if p == 1 else 'Норма' for p in preds_knn_new],
})
print(res_compare.to_string(index=False))

                      Описание  P LogisticRegression Прогноз LogisticRegression  P k-NN Прогноз k-NN
             Молодой, здоровый                 0.025                      Норма    0.02        Норма
         Пожилой, много рисков                 0.977                     Диабет    0.86       Диабет
Очень молодой, не обследовался                 0.009                      Норма    0.04        Норма
       Ср. возраст, лишний вес                 0.837                     Диабет    0.78       Диабет
       пьет алкоголь, активный                 0.048                      Норма    0.04        Норма
    Пожилая, инсульт, ожирение                 0.974                     Диабет    0.92       Диабет
   Ср. возраст, курит, активен                 0.394                      Норма    0.44        Норма
       Молодой, очень здоровый                 0.035                      Норма    0.02        Норма
                  Пожилой, ССЗ                 0.964                     Диабет    0.98    

In [40]:
# Сравнительный график на новых данных
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Логистическая регрессия', 'k-NN (лучший)'],
    horizontal_spacing=0.08)

for col, probs_new in enumerate([probs_lr, probs_knn_new], 1):
    bar_colors = px.colors.sample_colorscale('RdYlGn_r', probs_new.tolist())
    fig.add_trace(go.Bar(
        x=descriptions, y=probs_new,
        marker_color=bar_colors,
        text=[f'{p:.3f}' for p in probs_new],
        textposition='outside',
        showlegend=False
    ), row=1, col=col)
    fig.add_hline(y=0.5, line_dash='dash', line_color='magenta', row=1, col=col,
                  annotation_text='Порог 0.5', annotation_position='top right')

fig.update_xaxes(tickangle=-25)
fig.update_yaxes(range=[0, 1.15], title_text='P(Diabetes_binary = 1)')
fig.update_layout(height=500, width=1600,
    title='Рисунок 19. Вероятности на 10 составленных примерах: ЛР vs k-NN')
fig.show()

Цвет столбца — градиент от зелёного (низкий риск) до красного (высокий риск), порог 0,5 — граница между прогнозами «Норма» и «Диабет».

## Итоговые выводы

| Модель | ROC-AUC | F1 | Recall | Precision | Accuracy |
|---|---|---|---|---|---|
| LogisticRegression базовая | ~0,823 | ~0,750 | ~0,764 | ~0,737 | ~0,746 |
| LogisticRegression лучшая | ~0,823 | ~0,750 | ~0,764 | ~0,737 | ~0,746 |
| k-NN лучший | ~0,818 | ~0,749 | ~0,773 | ~0,727 | ~0,741 |

- **Логистическая регрессия** незначительно превосходит k-NN по всем метрикам
  на этом датасете. Причина — преимущественно линейные зависимости между признаками
  и целевой переменной (корреляции невысокие, но монотонные).
- **k-NN** чувствителен к «шумным» бинарным признакам (большинство из 21 признака — 0/1),
  из-за чего расстояние Евклида в таком пространстве теряет различимость.
- Обе модели уверенно ошибаются примерно в **25–26 % случаев**.
- ROC-AUC ≈ **0,82** при Accuracy ≈ **0,74** — модель хорошо ранжирует риски,
  но порог 0,5 не оптимален.
- В задаче скрининга диабета критичнее минимизировать **FN (пропущенные больные)**:
  стоит снизить порог классификации ниже 0,5 или оптимизировать по Recall вместо F1.
- Для существенного улучшения качества потребуются нелинейные модели
  (деревья, ансамбли) или расширение признакового пространства клиническими данными.